In [24]:
import pandas as pd

df = pd.read_csv("../data/used_cars_data.csv", nrows=500_000)  # sample first — full file is 3M rows
df.shape

C:\Users\mrsha\AppData\Local\Temp\ipykernel_19484\101329347.py:3: DtypeWarning: Columns (0: bed, 1: dealer_zip) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/used_cars_data.csv", nrows=500_000)  # sample first — full file is 3M rows


(500000, 66)

In [25]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 66 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   vin                      500000 non-null  str    
 1   back_legroom             475891 non-null  str    
 2   bed                      2962 non-null    str    
 3   bed_height               53841 non-null   str    
 4   bed_length               53841 non-null   str    
 5   body_type                497978 non-null  str    
 6   cabin                    8745 non-null    str    
 7   city                     500000 non-null  str    
 8   city_fuel_economy        419864 non-null  float64
 9   combine_fuel_economy     0 non-null       float64
 10  daysonmarket             500000 non-null  int64  
 11  dealer_zip               500000 non-null  object 
 12  description              486280 non-null  str    
 13  engine_cylinders         485244 non-null  str    
 14  engine_displace

In [26]:
df[['price', 'mileage', 'year', 'make_name', 'model_name', 'trim_name',
    'horsepower', 'body_type', 'has_accidents', 'is_new']].describe(include='all')

,price,mileage,year,make_name,model_name,trim_name,horsepower,body_type,has_accidents,is_new
count,5.000000e+05,4.766010e+05,500000.000000,500000,500000,481882,474039.000000,497978,268968,500000
unique,NaN,NaN,NaN,72,1125,6024,NaN,9,2,2
top,NaN,NaN,NaN,Ford,F-150,SE FWD,NaN,SUV / Crossover,False,False
freq,NaN,NaN,NaN,73818,15737,10710,NaN,258741,224234,262001
mean,3.007750e+04,2.980093e+04,2017.653596,NaN,NaN,NaN,242.335719,NaN,NaN,NaN
std,2.017657e+04,4.259697e+04,4.015752,NaN,NaN,NaN,86.295478,NaN,NaN,NaN
min,2.990000e+02,0.000000e+00,1915.000000,NaN,NaN,NaN,70.000000,NaN,NaN,NaN
25%,1.859500e+04,6.000000e+00,2017.000000,NaN,NaN,NaN,175.000000,NaN,NaN,NaN
50%,2.699500e+04,1.056100e+04,2019.000000,NaN,NaN,NaN,240.000000,NaN,NaN,NaN
75%,3.799800e+04,4.252800e+04,2020.000000,NaN,NaN,NaN,295.000000,NaN,NaN,NaN


In [27]:
df.isnull().sum().sort_values(ascending=False)

is_certified               500000
combine_fuel_economy       500000
vehicle_damage_category    500000
bed                        497038
cabin                      491255
is_oemcpo                  469855
is_cpo                     458011
bed_length                 446159
bed_height                 446159
owner_count                245798
salvage                    231032
theft_title                231032
frame_damaged              231032
fleet                      231032
has_accidents              231032
isCab                      231032
franchise_make              99958
torque                      81390
city_fuel_economy           80136
highway_fuel_economy        80136
power                       75297
main_picture_url            67119
interior_color              58969
major_options               35464
engine_displacement         25961
horsepower                  25961
width                       24109
back_legroom                24109
front_legroom               24109
height        

In [28]:
pd.set_option('display.max_rows', None)
df.isnull().sum().sort_values(ascending=False)


is_certified               500000
combine_fuel_economy       500000
vehicle_damage_category    500000
bed                        497038
cabin                      491255
is_oemcpo                  469855
is_cpo                     458011
bed_length                 446159
bed_height                 446159
owner_count                245798
salvage                    231032
theft_title                231032
frame_damaged              231032
fleet                      231032
has_accidents              231032
isCab                      231032
franchise_make              99958
torque                      81390
city_fuel_economy           80136
highway_fuel_economy        80136
power                       75297
main_picture_url            67119
interior_color              58969
major_options               35464
engine_displacement         25961
horsepower                  25961
width                       24109
back_legroom                24109
front_legroom               24109
height        

In [29]:
drop_cols = [
    'is_certified', 'combine_fuel_economy', 'vehicle_damage_category',
    'bed', 'cabin', 'bed_length', 'bed_height', 'is_oemcpo', 'isCab',
    'main_picture_url', 'listing_id', 'sp_id', 'sp_name', 'dealer_zip',
    'latitude', 'longitude', 'vin'
]
df = df.drop(columns=drop_cols)
df.shape

(500000, 49)

In [30]:
df = df[df['is_new'] == False].copy()
df.shape

(262001, 49)

In [31]:
history_cols = ['salvage', 'theft_title', 'frame_damaged', 'fleet', 'has_accidents']

for col in history_cols:
    df[col] = df[col].fillna(False)

df['owner_count'] = df['owner_count'].fillna(df['owner_count'].median())

In [32]:
numeric_impute_cols = [
    'horsepower', 'torque', 'power', 'highway_fuel_economy',
    'city_fuel_economy', 'engine_displacement'
]

for col in numeric_impute_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # convert strings to NaN
    df[col] = df[col].fillna(df[col].median())


In [33]:
required_cols = ['price', 'mileage', 'year', 'make_name', 'model_name']
df = df.dropna(subset=required_cols)
df.shape

(258933, 49)

In [34]:
df = df[
    (df['price'].between(500, 150_000)) &
    (df['year'] >= 1990) &
    (df['mileage'] <= 300_000)
]
df.shape

(257719, 49)

In [35]:
df[['price', 'mileage', 'year', 'horsepower']].describe()

,price,mileage,year,horsepower
count,257719.000000,257719.000000,257719.000000,257719.000000
mean,22142.229389,54734.184461,2015.570955,241.589844
std,12858.287855,43449.580626,3.758077,81.741091
min,500.000000,50.000000,1990.000000,70.000000
25%,13932.000000,24385.000000,2014.000000,175.000000
50%,19600.000000,39414.000000,2017.000000,240.000000
75%,28300.000000,77436.000000,2018.000000,290.000000
max,150000.000000,300000.000000,2021.000000,808.000000


In [36]:
import re

def scrub_description(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\[!@@.*?@@!\]", "", text)
    text = text.replace("\n", " ").strip()
    return text[:400]   # shortened from 1000

def build_full(row):
    parts = [
        f"{int(row['year'])} {row['make_name']} {row['model_name']} {row['trim_name'] or ''}".strip(),
        f"Body type: {row['body_type']}",
        f"Mileage: {int(row['mileage']):,} miles",
        f"Horsepower: {int(row['horsepower'])} hp",
        f"Fuel type: {row['fuel_type']}" if pd.notna(row['fuel_type']) else "",
        f"Transmission: {row['transmission_display']}" if pd.notna(row['transmission_display']) else "",
        f"Accident history: {'Yes' if row['has_accidents'] else 'No reported accidents'}",
        f"Frame damage: {'Yes' if row['frame_damaged'] else 'No'}",
        scrub_description(row['description']),
    ]
    return "\n".join(p for p in parts if p)

df['full'] = df.apply(build_full, axis=1)
print(df['full'].iloc[0])

2020 Land Rover Range Rover Evoque P300 R-Dynamic SE AWD
Body type: SUV / Crossover
Mileage: 254 miles
Horsepower: 296 hp
Fuel type: Gasoline
Transmission: 9-Speed Automatic Overdrive
Accident history: No reported accidents
Frame damage: No
4-Wheel Disc Brakes,A/C,ABS,Adjustable Steering Wheel,All Wheel Drive,Aluminum Wheels,Auto-Dimming Rearview Mirror,Automatic Headlights,Automatic Parking,Auxiliary Audio Input,Back-Up Camera,Bluetooth Connection,Brake Actuated Limited Slip Differential,Brake Assist,Bucket Seats,Cargo Shade,Child Safety Locks,Climate Control,Cross-Traffic Alert,Cruise Control,Daytime Running Lights,Driver Adjustabl


In [37]:
import sys
sys.path.append("..")  # so we can import car_pricer from notebooks/

from auto_pricer.car import Car

def na(val):
    return None if pd.isna(val) else val

def row_to_car(row):
    return Car(
        title=f"{int(row['year'])} {row['make_name']} {row['model_name']}",
        make=row['make_name'],
        model=row['model_name'],
        trim=na(row.get('trim')),
        year=int(row['year']),
        mileage=float(row['mileage']),
        price=float(row['price']),
        horsepower=na(row.get('horsepower')),
        body_type=na(row.get('body_type')),
        fuel_type=na(row.get('fuel_type')),
        transmission=na(row.get('transmission')),
        has_accidents=na(row.get('has_accidents')),
        frame_damaged=na(row.get('frame_damaged')),
        full=na(row.get('full')),
    )


cars = [row_to_car(row) for _, row in df.iterrows()]
len(cars)

257719

In [38]:
for i, car in enumerate(cars):
    car.id = i

cars[0]

<2020 Land Rover Range Rover Evoque = $84399.0>

In [39]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(cars, test_size=0.1, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

len(train), len(val), len(test)

(231947, 12886, 12886)

In [40]:
print(train[0].full)
print(train[0])

2008 Honda CR-V EX AWD
Body type: SUV / Crossover
Mileage: 201,000 miles
Horsepower: 166 hp
Fuel type: Gasoline
Transmission: 5-Speed Automatic
Accident history: Yes
Frame damage: No
Nice older CRV , runs and drives like new . Originally out of Connecticut.Grille Color - Chrome, Air Filtration, Floor Mat Material - Carpet, Floor Mats - Front, Front Air Conditioning, Front Air Conditioning Zones - Single, Floor Mats - Rear, Armrests - Rear Center Folding With Storage And Pass-Thru, Cargo Cover - Hard, Cruise Control, Cupholders - Front, Cupholders - Rear, Steering Wheel - Tilt 
title='2008 Honda CR-V' make='Honda' model='CR-V' trim=None year=2008 mileage=201000.0 price=5495.0 horsepower=166.0 body_type='SUV / Crossover' fuel_type='Gasoline' transmission='A' has_accidents=True frame_damaged=False full='2008 Honda CR-V EX AWD\nBody type: SUV / Crossover\nMileage: 201,000 miles\nHorsepower: 166 hp\nFuel type: Gasoline\nTransmission: 5-Speed Automatic\nAccident history: Yes\nFrame damage: N

In [41]:
from dotenv import load_dotenv
from huggingface_hub import login
import os

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [42]:
from pydantic import BaseModel

from datasets import Dataset, DatasetDict, load_dataset
from typing import Self

class Car(BaseModel):
    # ... (keep everything you already have) ...

    @staticmethod
    def push_to_hub(dataset_name: str, train: list["Car"], val: list["Car"], test: list["Car"]):
        DatasetDict({
            "train": Dataset.from_list([c.model_dump() for c in train]),
            "validation": Dataset.from_list([c.model_dump() for c in val]),
            "test": Dataset.from_list([c.model_dump() for c in test]),
        }).push_to_hub(dataset_name)

    @classmethod
    def from_hub(cls, dataset_name: str) -> tuple[list[Self], list[Self], list[Self]]:
        ds = load_dataset(dataset_name)
        return (
            [cls.model_validate(row) for row in ds["train"]],
            [cls.model_validate(row) for row in ds["validation"]],
            [cls.model_validate(row) for row in ds["test"]],
        )

In [43]:
username = "ShahidHKhan"  # replace with your actual HF username
dataset_name = f"{username}/cars_raw"

Car.push_to_hub(dataset_name, train, val, test)
print(f"Pushed to https://huggingface.co/datasets/{dataset_name}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  3.47ba/s]
Processing Files (1 / 1): 100%|██████████| 68.8MB / 68.8MB, 1.11MB/s  
New Data Upload: 100%|██████████| 68.8MB / 68.8MB, 1.11MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:52<00:00, 52.55s/ shards]
Setting num_proc from 1 back to 1 for the validation split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 29.09ba/s]
Processing Files (1 / 1): 100%|██████████| 3.89MB / 3.89MB,  292kB/s  
New Data Upload: 100%|██████████| 3.89MB / 3.89MB,  292kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:04<00:00,  4.28s/ shards]
Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:0

Pushed to https://huggingface.co/datasets/ShahidHKhan/cars_raw


In [44]:
loaded_train, loaded_val, loaded_test = Car.from_hub(dataset_name)
print(len(loaded_train), len(loaded_val), len(loaded_test))
print(loaded_train[0])

Generating test split: 100%|██████████| 12886/12886 [00:00<00:00, 572740.48 examples/s]


231947 12886 12886

